# Sabeel: pothole detector training

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all.

Choices, aimed at high test metrics with fast CPU inference on our API:
- **YOLO11s**: clearly more accurate than YOLOv8n, still around 100 ms on a CPU at 640 px. If the latency in cell 8 is too slow for us, set `MODEL = "yolo11n.pt"` and retrain.
- **Two data sources**: RDD2022 potholes (dashcam views from 6 countries) plus 300 close-up pothole photos from Hugging Face (Roboflow, CC BY 4.0; like a citizen's phone snap). Pothole-free road photos are added so the model learns what a normal road looks like.
- **Held-out test split** (70/15/15). The confidence threshold is tuned on **val**, and the final numbers come from **test**.

The dataset is built on Colab's local disk; weights, plots and metrics go to Google Drive (`MyDrive/sabeel`), so a disconnect doesn't lose them.
Start with a quick test (`EPOCHS = 1`), then set `EPOCHS = 100` (roughly 3–5 h on a T4; keep the tab open). If Colab disconnects, see **Resume** at the bottom.

In [ ]:
# 1. Settings, GPU check, Drive
MODEL = "yolo11s.pt"  # "yolo11n.pt" = faster, a bit less accurate
EPOCHS = 100          # use 1 for a quick pipeline test
IMGSZ = 640           # keep 640: matches ml-training/inference.py and keeps CPU latency low
NEG_RATIO = 0.15      # pothole-free road photos, as a share of pothole photos
RDD_URL = "https://ndownloader.figshare.com/files/38030910"   # official RDD2022 on figshare (13.3 GB); the old S3 link returns 403

import subprocess
assert subprocess.run(["nvidia-smi"]).returncode == 0, "No GPU: Runtime → Change runtime type → T4 GPU"

from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/sabeel"
!mkdir -p {SAVE}

In [ ]:
# 2. Install
!pip install -q -U ultralytics onnxruntime

In [ ]:
# 3. Get the data. Fast path: restore the built dataset cached on Drive by an earlier session (~1 min).
#    Slow path (first time only): download and unpack RDD2022 (13.3 GB from figshare, 15-40 min).
import os, glob
CACHE = f"{SAVE}/data_cache.zip"
CACHED = os.path.exists(CACHE)
if CACHED:
    !unzip -q -o "{CACHE}" -d /content
    print("restored dataset from Drive:", {s: len(glob.glob(f'/content/data/{s}/images/*')) for s in ('train', 'val', 'test')})
else:
    import os, glob
    def n_xml(): return len(glob.glob("/content/rdd/**/*.xml", recursive=True))
    if n_xml() == 0:                       # re-downloads if an earlier attempt failed or was cut short
        !rm -rf /content/rdd /content/rdd.zip
        !wget --show-progress -O /content/rdd.zip "{RDD_URL}"
        !ls -la /content/rdd.zip && file /content/rdd.zip
        !mkdir -p /content/rdd && unzip -q /content/rdd.zip -d /content/rdd && rm /content/rdd.zip
        # the archive contains one zip per country (maybe nested), unpack until none are left
        while zips := glob.glob("/content/rdd/**/*.zip", recursive=True):
            for z in zips:
                subprocess.run(["unzip", "-q", "-o", z, "-d", os.path.dirname(z)], check=True)
                os.remove(z)
    !df -h /content | tail -1
    !find /content/rdd -maxdepth 5 -type d | head -40
    print("XML annotation files:", n_xml(), "| images:", len(glob.glob("/content/rdd/**/*.jpg", recursive=True)))
    assert n_xml(), "No annotations: the download failed (see the wget / file output above)"

In [ ]:
# 4. Build one dataset with ml-training/build_dataset.py, the single source of truth (issue #45):
#    RDD2022 D40 + close-up pothole photos + negatives, split 70/15/15, seed 42.
#    The repo is private, so copy ml-training/build_dataset.py into MyDrive/sabeel/ once.
#    Skipped when cell 3 restored the dataset from Drive; after a fresh build it is cached to Drive.
if CACHED:
    print("dataset restored from Drive, nothing to build")
else:
    assert os.path.exists(f"{SAVE}/build_dataset.py"), "Copy ml-training/build_dataset.py to MyDrive/sabeel/ first"
    !pip install -q huggingface_hub
    !python "{SAVE}/build_dataset.py" --rdd /content/rdd --out /content/data --neg-ratio {NEG_RATIO} --seed 42
    !cp /content/data/data.yaml /content/data.yaml
    !cd /content && zip -q -r -0 "{CACHE}" data data.yaml
    print("cached dataset to Drive:", CACHE)


In [ ]:
# 5. Train (weights saved to Drive after every epoch)
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data="/content/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=-1, device=0,
            patience=25, cos_lr=True, close_mosaic=10, workers=2,
            project=f"{SAVE}/runs", name="pothole", exist_ok=True)

In [ ]:
# 6. Tune the confidence threshold on VAL (max F1), then score once on TEST
import json, numpy as np
best = YOLO(f"{SAVE}/runs/pothole/weights/best.pt")

v = best.val(data="/content/data.yaml", split="val", imgsz=IMGSZ, plots=False,
             project=f"{SAVE}/runs", name="val", exist_ok=True)
try:
    f1 = np.asarray(v.box.f1_curve).mean(0)
    BEST_CONF = round(float(np.asarray(v.box.px)[f1.argmax()]), 3)
    print("best confidence threshold on val:", BEST_CONF, "| F1:", round(float(f1.max()), 3))
except Exception as e:
    BEST_CONF = 0.25
    print("threshold tuning unavailable in this ultralytics version, using 0.25:", e)

m = best.val(data="/content/data.yaml", split="test", imgsz=IMGSZ,
             project=f"{SAVE}/runs", name="test", exist_ok=True)
try:
    tf1 = np.asarray(m.box.f1_curve).mean(0)
    test_f1 = round(float(tf1[int(np.abs(np.asarray(m.box.px) - BEST_CONF).argmin())]), 4)
except Exception:
    test_f1 = None
metrics = {
    "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ,
    "test_images": len(glob.glob("/content/data/test/images/*")),
    "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
    "mAP50": round(float(m.box.map50), 4), "mAP50-95": round(float(m.box.map), 4),
    "conf_threshold": BEST_CONF, "test_F1_at_threshold": test_f1,
}
print(metrics)

In [ ]:
# 7. Export to ONNX (fixed 640x640, same contract as ml-training/inference.py)
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
print("ONNX:", onnx_path)

In [ ]:
# 8. CPU latency of the ONNX model (Colab has 2 slow vCPUs, so our server will be at least this fast)
import time, onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
x = np.random.rand(1, 3, IMGSZ, IMGSZ).astype(np.float32)
for _ in range(3): sess.run(None, {inp.name: x})
t = []
for _ in range(20):
    t0 = time.perf_counter(); sess.run(None, {inp.name: x}); t.append((time.perf_counter() - t0) * 1000)
metrics["cpu_ms_colab"] = round(float(np.median(t)), 1)
metrics["onnx_output_shape"] = [int(s) for s in sess.run(None, {inp.name: x})[0].shape]   # expect [1, 5, 8400]
print(f"median CPU inference: {metrics['cpu_ms_colab']} ms | output {metrics['onnx_output_shape']}")
Path(f"{SAVE}/test_metrics.json").write_text(json.dumps(metrics, indent=2))
!ls -la {SAVE}/runs/pothole/weights {SAVE}

In [ ]:
# 9. Visual check: predictions on a few test images at the tuned threshold
from IPython.display import Image as Show, display
for p in sorted(glob.glob("/content/data/test/images/*.jpg"))[:8]:
    best.predict(p, conf=BEST_CONF, imgsz=IMGSZ, verbose=False)[0].save(filename="/content/pred.jpg")
    display(Show("/content/pred.jpg", width=480))

## Resume after a disconnect
**Only if a training run was cut off before it finished.** A finished run (like the 1-epoch test) can't be resumed: to train longer, change `EPOCHS` in cell 1 and run cell 5 again.

Run cells 1–4 (they restore the dataset from Drive in about a minute), then the cell below, then cells 6–9.

In [ ]:
import torch
from ultralytics import YOLO
LAST = f"{SAVE}/runs/pothole/weights/last.pt"
ckpt = torch.load(LAST, map_location="cpu", weights_only=False)
# a finished run has its optimizer stripped and epoch set to -1; resuming it makes ultralytics start an unrelated default training
if ckpt.get("optimizer") is None or ckpt.get("epoch", -1) < 0:
    print("That run already finished, so there is nothing to resume. Set EPOCHS in cell 1, rerun cell 1, then run cell 5.")
else:
    print(f"resuming from epoch {ckpt['epoch'] + 1}")
    YOLO(LAST).train(resume=True)

## Then
From Google Drive → `MyDrive/sabeel`, download:
- `runs/pothole/weights/best.onnx` (the new model) and `best.pt` (for retraining)
- `test_metrics.json` (the numbers for the README and the judges)
- `runs/pothole/results.png`, `runs/test/confusion_matrix.png`, `runs/test/PR_curve.png` (for the slides)